In [1]:
import scarf

scarf.set_verbosity('WARNING')

scarf.cytebase.connect("scarf_docs").download_dataset(
    'tenx_5K_pbmc_rnaseq',
    destination='scarf_datasets',
    zarr=True,
)
ds = scarf.DataStore(
    'scarf_datasets/tenx_5K_pbmc_rnaseq/data.zarr',
    nthreads=4,
    min_features_per_cell=10,
)
ds.filter_cells(
    attrs=['RNA_nCounts', 'RNA_nFeatures'],
    highs=[15000, 4000],
    lows=[1000, 500],
    reset_previous=True,
)
if 'I__hvgs' not in ds.RNA.feats.columns:
    ds.mark_hvgs(min_cells=20, top_n=500, show_plot=False)

In [2]:
normalized = ds.run_normalization(
    feat_key='hvgs',
    update_state=False,
)
pca = ds.run_pca(normalized, dims=15, update_state=False)
ann = ds.build_ann_index(pca, update_state=False)
neighbors_k11 = ds.query_neighbors(ann, k=11, update_state=False)
graph_k11 = ds.build_connectivity_map(neighbors_k11, update_state=False)

In [3]:
neighbors_k15 = ds.query_neighbors(ann, k=15, update_state=False)
graph_k15 = ds.build_connectivity_map(neighbors_k15, update_state=False)

assert ds.run_normalization(feat_key='hvgs', update_state=False) == normalized
assert ds.run_pca(normalized, dims=15, update_state=False) == pca
assert ds.build_ann_index(pca, update_state=False) == ann
assert neighbors_k15 != neighbors_k11
assert graph_k15 != graph_k11

In [4]:
pca_dims20 = ds.run_pca(normalized, dims=20, update_state=False)
ann_dims20 = ds.build_ann_index(pca_dims20, update_state=False)
neighbors_dims20 = ds.query_neighbors(ann_dims20, k=11, update_state=False)
graph_dims20 = ds.build_connectivity_map(neighbors_dims20, update_state=False)

assert pca_dims20 != pca
assert ann_dims20 != ann
assert neighbors_dims20 != neighbors_k11
assert graph_dims20 != graph_k11
assert ds.run_normalization(feat_key='hvgs', update_state=False) == normalized

In [5]:
forced = ds.run_normalization(
    feat_key='hvgs',
    update_state=False,
    invalidate_cache=True,
)
assert forced != normalized
status = ds.inspect_artifact(forced)
status.complete, status.operation

(True, 'run_normalization')

In [6]:
def walk_inputs(store, ref, prefix=''):
    status = store.inspect_artifact(ref)
    print(f'{prefix}{ref.kind} ({status.operation})')
    for name, value in (status.inputs or {}).items():
        if isinstance(value, dict) and value.get('type') == 'artifact':
            walk_inputs(
                store,
                scarf.ArtifactRef.from_dict(value),
                prefix=prefix + '  ',
            )
        else:
            print(f'{prefix}  {name}: {value}')

walk_inputs(ds, graph_k11)

connectivity_map (build_connectivity_map)
  neighbors (query_neighbors)
    ann_index (build_ann_index)
      reduction (run_pca)
        normalized (run_normalization)
          cell_selection (filter_cells)
            metadata_fingerprints: {'RNA_nCounts': 'a8a7520000e32d28bcf97a8977290bcc7185570098e1fe95739c74687b435843', 'RNA_nFeatures': 'a3df08addee7271194b0ebdfac85ad92ec93f3801031e65316776453eb200f9c'}
            ordered_row_ids_fingerprint: f5f05615ffc8833f75752b75152ccd728702f42950f81e9c28402880a4dd2ae5
            values_fingerprint: c56cd97011e0ca999890ff144c9e3b8c10121cd0ee887e0f69f9abd4512e68f2
          feature_selection (manual_selection)
            ordered_row_ids_fingerprint: 4548c9820ab9ed0afbc48cbb6503bdcdc725b57c65dbfe3c4da8bb480b7fc4b1
            values_fingerprint: d4983f2e5495a647b233a55decd66c95091eb37c82ad94efc56044279783a0cf
        feature_scaling (calculate_feature_scaling)
          normalized (run_normalization)
            cell_selection (filter_cells)